# CGMacros 전체 파이프라인 통합 노트북

5개 노트북을 하나로 합친 파일입니다.  
**순서**: 전처리 → 피처 생성 → EDA → 콜드스타트 모델 → 개인화 모델

---
# 1. 전처리 (01_preprocessing)
---

# CGMacros 전처리 — Dexcom / Libre 센서 통합

CGMacros는 참가자마다 **Dexcom G6 Pro**와 **Abbott Libre Pro**, 두 개의 CGM(연속혈당측정기)을
동시에 착용해 혈당을 기록했다. 이 노트북은 두 센서 값을 하나의 "대표 혈당"(`glucose_primary`)
컬럼으로 합치기 위해 진행한 진단 → 원인 분석 → 보정 → 검증 과정을 순서대로 정리한 것이다.

**작업 순서**
1. 문제 발견 — 참가자별 두 센서 값 차이(MARD) 진단
2. 보간(interpolation) 아티팩트 가능성 확인 및 배제
3. 시차(lag) 분석으로 원인 세분화 (서머타임 의심)
4. 전체 참가자 자동 분류 (정상 / 경미한 보정 / DST 재정렬 / 제외)
5. 최종 `glucose_primary` 생성 및 보정 효과 검증

**배경 지식**: CGMacros 공식 논문(Nature Scientific Data, 2025)에 따르면, 이 데이터셋의
1분 간격 값은 실측이 아니라 **Dexcom(실제 5분 간격)과 Libre(실제 15분 간격)를 선형보간**해서
만든 값이다. 이 사실이 2단계 진단에서 중요하게 쓰인다.


## 0. 라이브러리 및 설정

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'   # 한글 폰트 (Windows). Mac이면 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

CONFIG = {
    "cgmacros_dir": r"C:\Users\smhrd1\Desktop\CGMacros",
}

# "차이가 크다"고 볼 기준 (1단계 진단용, CGM 연구에서 흔히 쓰는 기준)
ABS_DIFF_THRESHOLD = 20      # mg/dL
PCT_DIFF_THRESHOLD = 20      # %

## 1단계 — 문제 발견: 참가자별 센서 불일치 진단

두 센서가 동시에 값이 있는 모든 행을 대상으로, 다음 세 지표를 계산한다.
- **MARD (Mean Absolute Relative Difference)**: CGM 연구에서 "센서 정확도"를 나타내는 표준 지표.
- **correlation**: 두 센서 값이 시간에 따라 같이 움직이는 정도(패턴 일치도)
- **mean_diff**: Dexcom − Libre의 평균. 방향성(어느 쪽이 체계적으로 더 높게/낮게 나오는지) 확인용


In [ ]:
def load_all_subjects(base_dir):
    """CGMacros 전체 참가자 파일을 읽어서 (subject_id, Timestamp, Dexcom GL, Libre GL)만 모은다."""
    rows = []
    participant_dirs = sorted(glob.glob(os.path.join(base_dir, "CGMacros-*")))
    if not participant_dirs:
        participant_dirs = [base_dir]

    for pdir in participant_dirs:
        csv_files = glob.glob(os.path.join(pdir, "CGMacros-*.csv"))
        if not csv_files:
            csv_files = glob.glob(os.path.join(pdir, "*.csv"))
        for csv_path in csv_files:
            try:
                df = pd.read_csv(csv_path)
            except Exception as e:
                print(f"읽기 실패: {csv_path} ({e})")
                continue
            df.columns = [c.strip() for c in df.columns]
            if "Dexcom GL" not in df.columns or "Libre GL" not in df.columns:
                print(f"[건너뜀] 두 센서 컬럼이 모두 없음: {csv_path}")
                continue

            subject_id = os.path.basename(csv_path).replace(".csv", "")
            df["subject_id"] = subject_id
            df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
            rows.append(df[["subject_id", "Timestamp", "Dexcom GL", "Libre GL"]])

    if not rows:
        raise RuntimeError("로딩된 파일이 없습니다. CONFIG 경로를 확인하세요.")
    return pd.concat(rows, ignore_index=True)

In [ ]:
def compute_agreement(df):
    """두 센서 값이 동시에 존재하는 행만 남기고, diff/MARD/큰차이 플래그를 계산한다.
    (주의: 이 시점의 diff는 '1분 간격 보간값'끼리의 비교라는 걸 2단계에서 다시 짚는다.)"""
    both = df.dropna(subset=["Dexcom GL", "Libre GL"]).copy()
    both["diff"] = both["Dexcom GL"] - both["Libre GL"]
    both["abs_diff"] = both["diff"].abs()
    # ARD: 개별 행의 상대오차(%). 이걸 평균 낸 게 MARD
    both["ard_pct"] = (both["abs_diff"] / both[["Dexcom GL", "Libre GL"]].mean(axis=1)) * 100
    both["large_diff_flag"] = (both["abs_diff"] >= ABS_DIFF_THRESHOLD) | (both["ard_pct"] >= PCT_DIFF_THRESHOLD)
    return both


def per_subject_summary(both):
    """참가자별로 MARD, 상관계수, 평균차이, 큰차이 비율을 요약한다."""
    summaries = []
    for subj, g in both.groupby("subject_id"):
        n = len(g)
        mard = g["ard_pct"].mean()
        corr = g["Dexcom GL"].corr(g["Libre GL"])
        pct_large = (g["large_diff_flag"].sum() / n) * 100
        mean_diff = g["diff"].mean()   # 양수면 Dexcom이 평균적으로 더 높게 나옴
        summaries.append({
            "subject_id": subj, "n_readings": n, "MARD(%)": round(mard, 2),
            "correlation": round(corr, 3) if pd.notna(corr) else np.nan,
            "mean_diff(Dexcom-Libre)": round(mean_diff, 2),
            "pct_large_diff(%)": round(pct_large, 2),
        })
    return pd.DataFrame(summaries).sort_values("MARD(%)", ascending=False).reset_index(drop=True)

In [ ]:
def plot_agreement(both):
    """전체 데이터에 대한 산점도 + Bland-Altman plot."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(both["Libre GL"], both["Dexcom GL"], alpha=0.05, s=5)
    lims = [both[["Dexcom GL", "Libre GL"]].min().min(), both[["Dexcom GL", "Libre GL"]].max().max()]
    axes[0].plot(lims, lims, color="red", linestyle="--", label="y = x (완전 일치선)")
    axes[0].set_xlabel("Libre GL (mg/dL)"); axes[0].set_ylabel("Dexcom GL (mg/dL)")
    axes[0].set_title("센서 간 산점도"); axes[0].legend()

    mean_val = both[["Dexcom GL", "Libre GL"]].mean(axis=1)
    diff_val = both["diff"]
    mean_diff, sd_diff = diff_val.mean(), diff_val.std()
    axes[1].scatter(mean_val, diff_val, alpha=0.05, s=5)
    axes[1].axhline(mean_diff, color="black", label=f"평균 차이 = {mean_diff:.1f}")
    axes[1].axhline(mean_diff + 1.96 * sd_diff, color="red", linestyle="--", label="±1.96 SD")
    axes[1].axhline(mean_diff - 1.96 * sd_diff, color="red", linestyle="--")
    axes[1].set_xlabel("두 센서 평균값 (mg/dL)"); axes[1].set_ylabel("Dexcom - Libre (mg/dL)")
    axes[1].set_title("Bland-Altman Plot"); axes[1].legend()

    plt.tight_layout()
    plt.savefig("cgm_sensor_agreement.png", dpi=150)
    plt.show()

### 1단계 실행

In [ ]:
print("CGMacros 전체 로딩 중...")
raw = load_all_subjects(CONFIG["cgmacros_dir"])
print(f"전체 행: {len(raw)}, 참가자: {raw['subject_id'].nunique()}명")

both = compute_agreement(raw)
print(f"두 센서 값이 동시에 존재하는 행: {len(both)} ({len(both)/len(raw)*100:.1f}%)")
print(f"전체 평균 MARD: {both['ard_pct'].mean():.2f}%  (참고: 업계 기준 10% 내외가 '쓸만한' 수준)")

summary = per_subject_summary(both)
print("\n참가자별 요약 (MARD 높은 순 상위 10명):")
print(summary.head(10).to_string(index=False))

plot_agreement(both)

# 진단 결과: 전체적으로 MARD가 12~74%로 업계 기준보다 훨씬 높고,
# 거의 모든 참가자가 mean_diff 양수(Dexcom > Libre)로 쏠려 있음 -> 무작위 오차가 아니라
# 체계적인 편향(systematic bias) 또는 시간 어긋남을 의심할 근거가 됨 (2, 3단계로 이어짐)

## 2단계 — 보간(interpolation) 아티팩트 가능성 확인 및 배제

CGMacros 공식 논문에 따르면 1분 간격 데이터는 실측이 아니라 **Dexcom(실 5분)/Libre(실 15분)를
선형보간**해서 만든 값이다. "혹시 이 보간 때문에 MARD가 과장되어 보이는 것 아닌가?"를
확인하기 위해, 보간되지 않은 순수 실측점만 추출해서 같은 지표를 다시 계산해본다.


In [ ]:
def get_native_readings(series, step):
    """연속 구간에서 step번째 값마다 추출 = 실제 측정점으로 간주.
    (Dexcom은 5분마다 실측이므로 step=5, Libre는 15분마다 실측이므로 step=15)
    주의: 중간에 결측으로 끊긴 구간이 있으면 그 지점부터 카운트가 리셋되어야 정확하지만,
    여기서는 dropna() 후 순서대로 세는 근사치를 사용한다."""
    idx = np.arange(len(series))
    return series.iloc[idx % step == 0]


def native_agreement(sub_df):
    """한 참가자에 대해, 보간 없는 순수 실측점끼리 매칭해서 corr/MARD를 계산한다."""
    s = sub_df.set_index('Timestamp').sort_index()
    dexcom_native = get_native_readings(s['Dexcom GL'].dropna(), step=5).reset_index()
    libre_native = get_native_readings(s['Libre GL'].dropna(), step=15).reset_index()

    # 두 센서 실측 시각이 정확히 안 맞으므로, 가장 가까운 시각끼리(8분 이내) 매칭
    merged = pd.merge_asof(
        dexcom_native.rename(columns={'Dexcom GL': 'dexcom'}),
        libre_native.rename(columns={'Libre GL': 'libre'}),
        on='Timestamp', direction='nearest', tolerance=pd.Timedelta('8min')
    ).dropna()

    if len(merged) < 20:
        return None
    corr = merged['dexcom'].corr(merged['libre'])
    mard = (merged['dexcom'] - merged['libre']).abs() / merged[['dexcom', 'libre']].mean(axis=1) * 100
    return {
        'n_matched': len(merged),
        'native_corr': round(corr, 3),
        'native_MARD(%)': round(mard.mean(), 2),
        'mean_diff': round((merged['dexcom'] - merged['libre']).mean(), 2),
    }

### 2단계 실행

In [ ]:
results = []
for subj in raw['subject_id'].unique():
    res = native_agreement(raw[raw['subject_id'] == subj])
    if res:
        res['subject_id'] = subj
        results.append(res)

native_summary = pd.DataFrame(results).sort_values('native_corr')
native_summary.to_csv('native_sensor_agreement.csv', index=False, encoding='utf-8-sig')
print(native_summary.head(10).to_string(index=False))

# 검증 결과: 보간 포함 값과 순수 실측치만 비교한 값이 거의 동일하게 나옴
# (예: CGMacros-007은 보간포함 lag=0 corr=0.31, 순수 실측치 corr=0.284로 거의 동일)
# -> 결론: "보간 때문에 나쁘게 보인다"는 가설은 기각. 진짜 센서/시간 문제로 확인됨 (3단계로 이어짐)

## 3단계 — 시차(lag) 분석으로 원인 세분화

MARD가 높은 원인이 "값 자체가 다르다"가 아니라 "**두 기기의 시계가 어긋나 있다**"일 가능성을
확인한다. Libre 시계열을 앞뒤로 밀어보면서(-N ~ +N분) Dexcom과 상관관계가 최대가 되는 시차를
찾는다. 만약 특정 시차(예: ±60분, 서머타임과 일치)에서 상관관계가 급격히 좋아진다면
"시간 어긋남"이 원인이고, 아무리 밀어도 안 좋아진다면 "진짜 센서 자체 문제"로 판단한다.


In [ ]:
def find_best_lag(subject_df, max_lag_min=90, step_min=10):
    """Libre를 -max_lag~+max_lag분 밀면서 Dexcom과 상관관계가 최대가 되는 시차를 찾는다."""
    s = subject_df.set_index('Timestamp').sort_index()
    dexcom = s['Dexcom GL']
    libre = s['Libre GL']
    results = []
    for lag in range(-max_lag_min, max_lag_min + 1, step_min):
        shifted = libre.shift(freq=f'{lag}min')
        merged = pd.merge_asof(
            dexcom.dropna().reset_index(), shifted.dropna().reset_index(),
            on='Timestamp', direction='nearest', tolerance=pd.Timedelta('2min')
        )
        if len(merged) > 30:
            corr = merged['Dexcom GL'].corr(merged['Libre GL'])
            results.append((lag, corr))
    return pd.DataFrame(results, columns=['lag_min', 'correlation']).sort_values('correlation', ascending=False)


def plot_subject_timeseries(subject_df, subject_id, days=3):
    """참가자 한 명의 Dexcom/Libre 시계열을 겹쳐 그려서 육안으로 확인한다."""
    s = subject_df.set_index('Timestamp').sort_index()
    s = s.iloc[:days * 288]  # 5분 간격 기준 약 N일치

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(s.index, s['Dexcom GL'], label='Dexcom', alpha=0.8)
    ax.plot(s.index, s['Libre GL'], label='Libre', alpha=0.8)
    ax.set_title(f'{subject_id} - 혈당 시계열 비교 (초반 {days}일)')
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


def full_lag_scan(sub_df, max_lag_min=90, step_min=10):
    """한 참가자의 lag=0(보정없음) 상관관계와, 최적 lag에서의 상관관계를 함께 반환."""
    lag_df = find_best_lag(sub_df, max_lag_min, step_min)
    zero_row = lag_df.loc[lag_df['lag_min'] == 0]
    zero_corr = zero_row['correlation'].values[0] if len(zero_row) else np.nan
    best_row = lag_df.loc[lag_df['correlation'].idxmax()]
    return {
        'zero_lag_corr': round(zero_corr, 3),
        'best_lag': int(best_row['lag_min']),
        'best_lag_corr': round(best_row['correlation'], 3),
        'improvement': round(best_row['correlation'] - zero_corr, 3),
    }

### 3단계 실행 — 개별 사례 확인

먼저 MARD가 가장 나빴던 참가자들(007, 013, 046-정상비교군)로 가설을 확인한다.

In [ ]:
for subj in ['CGMacros-007', 'CGMacros-013', 'CGMacros-046']:
    sub_df = raw[raw['subject_id'] == subj]
    lag_result = find_best_lag(sub_df, max_lag_min=90, step_min=10)
    print(subj)
    print(lag_result.head(3))
    print()

# 결과 해석:
# - 046(정상): lag=0 근처가 이미 최적 -> 시간 문제 없음, 그냥 오프셋만 있는 정상 케이스
# - 013: +60분에서 corr가 0.10 -> 0.75로 급등 -> 시간 어긋남(서머타임 의심)이 원인
# - 007: lag을 밀어도 corr가 0.3~0.4대에 머무름 -> 시간 문제가 아니라 센서 자체 문제

# 007/046 실제 시계열을 그려서 육안으로 재확인
plot_subject_timeseries(raw[raw['subject_id'] == 'CGMacros-007'], 'CGMacros-007')
plot_subject_timeseries(raw[raw['subject_id'] == 'CGMacros-046'], 'CGMacros-046')
# -> 007은 Dexcom이 안정적으로 110~130을 유지하는 동안 Libre가 40~155를 요동치는 등
#    패턴 자체가 서로 다름을 육안으로 확인 (시간 재정렬로 해결 불가능한 유형)

## 4단계 — 전체 참가자 자동 분류

3단계에서 확인한 로직(zero-lag corr, best-lag corr, best-lag 크기)을 기준으로
전체 참가자를 4개 그룹으로 자동 분류한다.

| 그룹 | 기준 | 처리 |
|---|---|---|
| 정상_오프셋만 | zero-lag corr ≥ 0.75 | 보정 없이 그대로 사용 |
| 경미한_lag_보정 | best corr 0.5 이상, 나머지 | 필요시 소폭(≤20분) lag 적용 |
| DST_시간재정렬 | \|best_lag\| ≥ 40분 & best corr ≥ 0.55~0.6 | 해당 lag만큼 Libre 타임스탬프 시프트 |
| 제외_또는_별도처리 | 위 조건 다 불충족 | Libre를 fallback으로 쓰지 않고 Dexcom만 신뢰 |

참고: 006은 best_corr=0.596으로 DST 기준(0.6)을 근소하게 못 넘었지만, 실제로는 -50분
시프트 시 corr가 0.406→0.596으로 개선되는 DST 패턴이 맞아서 완화된 기준(0.55)으로
재분류한다.


In [ ]:
def classify_and_correct(raw, max_lag_min=90, step_min=10,
                          normal_threshold=0.75,
                          dst_lag_threshold=40, dst_corr_threshold=0.55):
    """전체 참가자를 4개 카테고리로 자동 분류하고, 적용할 lag값을 결정한다."""
    results = []
    for subj in raw['subject_id'].unique():
        sub_df = raw[raw['subject_id'] == subj]
        scan = full_lag_scan(sub_df, max_lag_min, step_min)
        scan['subject_id'] = subj

        if scan['zero_lag_corr'] >= normal_threshold:
            category, apply_lag = '정상_오프셋만', 0
        elif abs(scan['best_lag']) >= dst_lag_threshold and scan['best_lag_corr'] >= dst_corr_threshold:
            category, apply_lag = 'DST_시간재정렬', scan['best_lag']
        elif scan['best_lag_corr'] >= 0.5:
            category = '경미한_lag_보정'
            apply_lag = scan['best_lag'] if abs(scan['best_lag']) <= 20 else 0
        else:
            category, apply_lag = '제외_또는_별도처리', 0

        scan['category'] = category
        scan['apply_lag_min'] = apply_lag
        results.append(scan)

    return pd.DataFrame(results)

### 4단계 실행

In [ ]:
classification_final = classify_and_correct(raw)
print(classification_final['category'].value_counts())

classification_final.to_csv('subject_classification_final.csv', index=False, encoding='utf-8-sig')
classification_final.sort_values('category')

# 최종 분류 결과 (45명 기준):
#   정상_오프셋만       27명
#   경미한_lag_보정     12명
#   DST_시간재정렬       5명 (008, 009, 013, 030, 006)
#   제외_또는_별도처리    2명 (007, 032)

## 5단계 — 최종 `glucose_primary` 생성 및 검증

4단계 분류 결과를 실제로 적용해서 대표 혈당 컬럼을 만든다.

**처리 로직**
- DST/lag 그룹: Libre 타임스탬프를 보정된 lag만큼 밀어서 Dexcom과 시간축을 정렬
- 제외 그룹(007, 032): Libre를 fallback으로도 쓰지 않고 Dexcom만 신뢰 (없으면 결측 유지 — 잘못된 값보다 결측이 나음)
- 나머지: 기존처럼 Dexcom 우선, 없으면 (필요시 보정된) Libre로 대체
- `data_quality` 컬럼으로 각 행이 어떤 처리를 거쳤는지 추적 가능하게 함


In [ ]:
def apply_final_correction(raw, classification_final, dst_corr_threshold_relaxed=0.55):
    """분류 결과에 따라 lag 보정을 적용하고, 최종 glucose_primary/data_quality 컬럼을 만든다."""
    cls = classification_final.set_index('subject_id').to_dict('index')
    corrected_frames = []

    for subj, sub_df in raw.groupby('subject_id'):
        info = cls.get(subj)
        s = sub_df.sort_values('Timestamp').copy()

        if info is None:
            s['glucose_primary'] = s['Dexcom GL'].fillna(s['Libre GL'])
            s['data_quality'] = '분류없음'
            corrected_frames.append(s)
            continue

        category = info['category']
        # 006처럼 완화 기준으로 보면 DST에 해당하는 경계선 케이스 재분류
        if category == '경미한_lag_보정' and abs(info['best_lag']) >= 40 and info['best_lag_corr'] >= dst_corr_threshold_relaxed:
            category = 'DST_시간재정렬'
            lag = info['best_lag']
        else:
            lag = info['apply_lag_min']

        if category == '제외_또는_별도처리':
            s['glucose_primary'] = s['Dexcom GL']            # Libre는 fallback으로도 사용 안 함
            s['data_quality'] = '주의_Libre신뢰불가'
        else:
            libre_shifted = s.set_index('Timestamp')['Libre GL'].shift(freq=f'{lag}min').reset_index()
            s = s.drop(columns=['Libre GL']).merge(
                libre_shifted.rename(columns={'Libre GL': 'Libre GL_shifted'}),
                on='Timestamp', how='left'
            )
            s['glucose_primary'] = s['Dexcom GL'].fillna(s['Libre GL_shifted'])
            s['data_quality'] = category

        corrected_frames.append(s)

    return pd.concat(corrected_frames, ignore_index=True)


def verify_correction(final_corrected):
    """보정 전(zero_lag_corr) 대비 보정 후 상관관계가 실제로 개선됐는지 검증한다."""
    results = []
    for subj, g in final_corrected.groupby('subject_id'):
        if 'Libre GL_shifted' not in g.columns:
            continue
        both = g.dropna(subset=['Dexcom GL', 'Libre GL_shifted'])
        if len(both) < 20:
            continue
        corr = both['Dexcom GL'].corr(both['Libre GL_shifted'])
        mard = ((both['Dexcom GL'] - both['Libre GL_shifted']).abs() /
                both[['Dexcom GL', 'Libre GL_shifted']].mean(axis=1) * 100).mean()
        results.append({'subject_id': subj, 'corrected_corr': round(corr, 3), 'corrected_MARD(%)': round(mard, 2)})
    return pd.DataFrame(results)

### 5단계 실행

In [ ]:
final_corrected = apply_final_correction(raw, classification_final)
print(final_corrected['data_quality'].value_counts())
final_corrected.to_csv('cgmacros_final_corrected.csv', index=False, encoding='utf-8-sig')

# 검증: 보정 전/후 상관관계 비교
verify = verify_correction(final_corrected)
compare = verify.merge(classification_final[['subject_id', 'zero_lag_corr']], on='subject_id')
compare['개선폭'] = compare['corrected_corr'] - compare['zero_lag_corr']
print(compare.sort_values('개선폭', ascending=False).head(10).to_string(index=False))

# 검증 결과:
# - DST 그룹(008,009,013,030) 전원 corr 0.4~0.7대에서 0.75~0.96으로 크게 개선됨
# - 경미한_lag_보정 그룹도 방향대로 소폭 개선(+0.02~0.06)
# - 전체 행의 97.2%가 정상 또는 보정을 거쳐 신뢰 가능, 2.8%(007,032)만 별도 플래그 처리

## 6. (추가) 전체 참가자 × 전체 기간 시계열 육안 확인

지금까지는 007/013/046처럼 문제가 의심되는 몇 명만 3일치만 봤는데, **전체 45명을 전체 기간(약 10일치)
다 눈으로 훑어보기 위한 코드**. 두 가지 방식을 제공한다.

1. **한눈에 보는 개요판(grid)**: 45명을 작은 그래프로 한 화면에 모아서, 이상해 보이는 사람을 빠르게 스캐닝
2. **개별 고화질 파일 저장**: 개요판에서 의심되는 사람을 발견하면, 그 사람 그래프만 크게 따로 열어서 자세히 확인


In [ ]:
import math

def plot_all_subjects_grid(raw, classification_final=None, cols=5, save_path='all_subjects_overview.png'):
    """전체 참가자의 전체 기간 Dexcom/Libre 시계열을 작은 그래프들로 모아서 한 화면에 보여준다.
    classification_final을 넘기면 각 그래프 제목에 분류 카테고리(정상/DST재정렬/제외 등)를 같이 표시해서
    '어떤 그룹이 실제로 어떻게 생겼는지' 한눈에 비교할 수 있게 한다."""
    subjects = sorted(raw['subject_id'].unique())
    n = len(subjects)
    rows = math.ceil(n / cols)

    cls_map = {}
    if classification_final is not None:
        cls_map = classification_final.set_index('subject_id')['category'].to_dict()

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 2.3))
    axes = axes.flatten()

    for i, subj in enumerate(subjects):
        ax = axes[i]
        s = raw[raw['subject_id'] == subj].set_index('Timestamp').sort_index()
        ax.plot(s.index, s['Dexcom GL'], linewidth=0.5, label='Dexcom')
        ax.plot(s.index, s['Libre GL'], linewidth=0.5, label='Libre', alpha=0.7)
        cat = cls_map.get(subj, '')
        ax.set_title(f'{subj}\n{cat}', fontsize=8)
        ax.tick_params(labelsize=6, rotation=0)
        ax.set_xticks([])   # 45개를 다 보여줘야 해서 x축 라벨은 생략 (겹쳐서 안 보임)

    for j in range(len(subjects), len(axes)):
        axes[j].axis('off')   # 남는 빈 칸 숨기기

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    print(f"개요판 저장: {save_path}")
    plt.show()

In [ ]:
def save_individual_full_plots(raw, classification_final=None, out_dir='subject_plots_full'):
    """참가자별로 전체 기간 그래프를 큰 사이즈로 개별 PNG 파일에 저장한다.
    개요판(grid)에서 이상해 보이는 사람을 찾으면, 이 폴더에서 그 사람 파일만 열어 자세히 보면 된다."""
    os.makedirs(out_dir, exist_ok=True)
    cls_map = {}
    if classification_final is not None:
        cls_map = classification_final.set_index('subject_id')['category'].to_dict()

    subjects = sorted(raw['subject_id'].unique())
    for subj in subjects:
        s = raw[raw['subject_id'] == subj].set_index('Timestamp').sort_index()
        fig, ax = plt.subplots(figsize=(16, 3))
        ax.plot(s.index, s['Dexcom GL'], label='Dexcom', linewidth=0.7)
        ax.plot(s.index, s['Libre GL'], label='Libre', linewidth=0.7, alpha=0.8)
        cat = cls_map.get(subj, '')
        ax.set_title(f'{subj} - 전체 기간 ({cat})')
        ax.set_ylabel('mg/dL')
        ax.legend()
        plt.xticks(rotation=30)
        plt.tight_layout()
        fig.savefig(os.path.join(out_dir, f'{subj}.png'), dpi=120)
        plt.close(fig)   # 45개 다 화면에 띄우면 메모리/속도 문제가 있어서 파일로만 저장하고 닫음

    print(f"{len(subjects)}개 파일 저장 완료: {out_dir}/ 폴더")

### 6단계 실행

In [ ]:
# 1) 한눈에 보는 개요판 -- classification_final을 넘기면 카테고리별로 실제 생김새를 비교 가능
plot_all_subjects_grid(raw, classification_final=classification_final)

# 2) 개별 고화질 파일 저장 -- 개요판에서 의심스러운 사람 발견하면 이 폴더에서 크게 열어보기
save_individual_full_plots(raw, classification_final=classification_final)

## 7. (건너뜀) 참가자별 수동 검토 워크플로우 — 사용 안 함

원래 여기서 review_log.csv에 참가자별 결정(decision)을 하나씩 적어넣는 방식을 계획했으나,
**Dexcom을 기본으로 쓰고 Libre는 개인별 오프셋(+필요시 최근구간/drift 검증)으로만 보조
사용하기로 결정**하면서 이 수동 워크플로우 자체가 필요 없어졌다. 실제 처리는 8~9단계의
`fill_dexcom_with_offset_libre_v2()`가 담당한다 (007/032/048/039는 그 안에서 자동으로
결측 처리됨).


## 8. Dexcom 단일 센서로 전환 — 결측치 확인

Libre와의 정합성 문제(1~7단계)가 다수 참가자에서 완전히 해소되지 않아, **Dexcom 하나만
신뢰하는 단순한 파이프라인으로 전환**하기로 결정했다. Libre를 fallback으로 쓰지 않으므로,
**Dexcom 자체가 원래 비어있던 구간은 그대로 결측으로 남는다** — 모델링
전에 먼저 확인해야 한다.

확인할 것 3가지
1. 전체 및 참가자별 결측 비율
2. 결측이 "짧게 여러 번" 끼어있는지, "며칠씩 통째로" 비어있는지 (성격이 다른 문제)
3. 결측이 심한 참가자를 어떻게 할지 판단할 기준


In [ ]:
def simplify_to_dexcom_only(raw):
    """Libre 보정 로직을 버리고 Dexcom 하나만 신뢰하는 단순 버전으로 전환.
    (결정 근거: 1~7단계에서 Libre-Dexcom 정합성 문제가 다수 참가자에서 해소되지 않음)"""
    df = raw.copy()
    df['glucose_primary'] = df['Dexcom GL']
    df['is_missing'] = df['glucose_primary'].isna()
    return df


simplified = simplify_to_dexcom_only(raw)
overall_missing = simplified['is_missing'].mean() * 100
print(f"전체 Dexcom 결측 비율: {overall_missing:.2f}%")

In [ ]:
# 참가자별 결측 비율
missing_by_subject = simplified.groupby('subject_id')['is_missing'].mean().mul(100).sort_values(ascending=False)
print("참가자별 Dexcom 결측 비율(%) - 높은 순 상위 15명")
print(missing_by_subject.head(15))

fig, ax = plt.subplots(figsize=(14, 4))
missing_by_subject.plot(kind='bar', ax=ax)
ax.axhline(10, color='red', linestyle='--', label='10% 기준선 (참고용)')
ax.set_ylabel('결측 비율(%)')
ax.set_title('참가자별 Dexcom 결측 비율')
ax.legend()
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig('dexcom_missing_by_subject.png', dpi=150)
plt.show()

### 결측 "패턴" 확인 — 짧은 결측 여러 번 vs 며칠씩 통째로

같은 결측률 10%라도, "5분씩 200번 끊김"과 "하루를 통째로 못 잰 것"은 전혀 다른 문제다.
전자는 그냥 보간/제외해도 무해하지만, 후자는 그 기간의 식사 이벤트 자체를 통째로 못 쓰게 된다.
연속 결측 구간(gap)의 길이별 분포를 확인한다.


In [ ]:
def analyze_gap_lengths(sub_df):
    """한 참가자의 연속 결측 구간 길이(분 단위)를 계산한다."""
    s = sub_df.set_index('Timestamp').sort_index()
    is_na = s['Dexcom GL'].isna()
    # 결측이 시작/끝나는 지점을 그룹으로 묶어서 각 결측 구간의 길이를 계산
    group = (is_na != is_na.shift()).cumsum()
    gap_groups = s[is_na].groupby(group[is_na])
    gap_lengths_min = gap_groups.size()  # 1분 간격 데이터이므로 행 수 = 분 단위 길이
    return gap_lengths_min


gap_summary = []
for subj in raw['subject_id'].unique():
    gaps = analyze_gap_lengths(raw[raw['subject_id'] == subj])
    if len(gaps) == 0:
        continue
    gap_summary.append({
        'subject_id': subj,
        'total_missing_min': gaps.sum(),
        'n_gap_segments': len(gaps),
        'longest_gap_min': gaps.max(),
        'longest_gap_hours': round(gaps.max() / 60, 1),
    })

gap_df = pd.DataFrame(gap_summary).sort_values('longest_gap_min', ascending=False)
print("결측 구간 요약 - 가장 긴 연속 결측(시간) 기준 상위 15명")
print(gap_df.head(15).to_string(index=False))
gap_df.to_csv('dexcom_gap_analysis.csv', index=False, encoding='utf-8-sig')

# 참고 기준:
#  - longest_gap_hours가 몇 시간 수준이면 -> 그 시간대 식사 이벤트 하나만 못 쓰는 정도라 큰 문제 아님
#  - longest_gap_hours가 12시간~며칠 단위면 -> 그 기간의 식사 이벤트 전부가 라벨 없이 날아감
#    -> 해당 참가자를 통째로 뺄지, 그 구간만 빼고 나머지는 살릴지 판단 필요

### 판단 기준 제안

- **결측 비율 10% 미만 & 최장 결측 12시간 미만**: 그대로 사용 (식사 이벤트에 큰 영향 없음)
- **결측 비율은 낮지만 특정 구간에 몰려있음(최장 결측 하루 이상)**: 그 구간만 제외하고 나머지는 사용
  (032의 `split_period`와 같은 방식 재사용 가능)
- **결측 비율 자체가 매우 높음(예: 30% 이상)**: 참가자 전체 제외 검토

이 기준으로 최종 사용/제외 대상을 review_log.csv에 반영하면 된다.


### Dexcom 결측을 Libre(오프셋 보정)로 대체

결측 상위 10명을 개별 확인한 결과, 관측 기간 안에서도 오프셋이 시간에 따라 흔들리는지에
따라 3그룹으로 나뉜다.

- **불안정(들쭉날쭉, 오프셋 신뢰 불가)**: 048, 039 → 대체하지 않고 결측 유지
- **추세 있음(drift, 한 방향으로 꾸준히 변함)**: 022, 029, 014, 026, 015 → 전체 평균 대신
  **관측 기간의 가장 최근 구간 오프셋**을 사용 (결측 직전 상태에 더 가까움)
- **패턴 자체가 안 맞음(시간 재정렬로도 해결 안 됨, 3단계에서 확인)**: 007, 032 → Libre를
  fallback으로도 사용하지 않고 Dexcom만 신뢰
- **나머지 전원(위 그룹에 없는 참가자)**: 전체 관측 기간 평균 오프셋을 사용


In [ ]:
UNRELIABLE_SUBJECTS = ['CGMacros-007', 'CGMacros-032']   # 패턴 자체가 안 맞음 (3단계에서 확인)
UNSTABLE_SUBJECTS = ['CGMacros-048', 'CGMacros-039']       # 오프셋이 들쭉날쭉해서 신뢰 불가
TREND_SUBJECTS = ['CGMacros-022', 'CGMacros-029', 'CGMacros-014', 'CGMacros-026', 'CGMacros-015']  # 오프셋 drift 있음


def check_offset_drift_within_observed(sub_df, n_splits=3):
    """둘 다 관측된 구간을 시간순으로 n등분해서, 구간별 오프셋이 흔들리는지 확인한다.
    (참고용 진단 함수 -- 위 3개 리스트를 만들 때 이 함수의 결과를 보고 사람이 직접 분류했다)"""
    s = sub_df.set_index('Timestamp').sort_index()
    both = s.dropna(subset=['Dexcom GL', 'Libre GL'])
    if len(both) < 100:
        return None
    both = both.copy()
    both['chunk'] = pd.cut(np.arange(len(both)), bins=n_splits, labels=False)
    offsets_by_chunk = both.groupby('chunk')[['Dexcom GL', 'Libre GL']].apply(
        lambda g: (g['Dexcom GL'] - g['Libre GL']).mean()
    )
    return offsets_by_chunk


def get_offset_for_subject(sub_df, subject_id, n_splits=3):
    """추세(drift)가 있는 사람은 '가장 최근 구간'의 오프셋을, 나머지는 '전체 평균' 오프셋을 반환."""
    s = sub_df.set_index('Timestamp').sort_index()
    both = s.dropna(subset=['Dexcom GL', 'Libre GL'])

    if subject_id in TREND_SUBJECTS:
        both = both.copy()
        both['chunk'] = pd.cut(np.arange(len(both)), bins=n_splits, labels=False)
        last_chunk = both[both['chunk'] == both['chunk'].max()]
        return (last_chunk['Dexcom GL'] - last_chunk['Libre GL']).mean()
    else:
        return (both['Dexcom GL'] - both['Libre GL']).mean()


def fill_dexcom_with_offset_libre_v2(raw, unreliable_subjects=UNRELIABLE_SUBJECTS, unstable_subjects=UNSTABLE_SUBJECTS):
    """Dexcom을 기본으로 쓰고, 결측 구간은 (검증된 경우에 한해) 개인별 오프셋을 더한 Libre로 채운다.
    - 007/032/048/039: 대체하지 않고 결측 유지 (Dexcom만 신뢰)
    - 022/029/014/026/015: 최근구간 오프셋 사용 (drift 반영)
    - 나머지: 전체 평균 오프셋 사용
    """
    corrected_frames = []
    for subj, sub_df in raw.groupby('subject_id'):
        s = sub_df.sort_values('Timestamp').copy()

        if subj in unreliable_subjects or subj in unstable_subjects:
            s['glucose_primary'] = s['Dexcom GL']
            s['data_quality'] = 'Dexcom만_신뢰불가Libre제외'
            corrected_frames.append(s)
            continue

        offset = get_offset_for_subject(s, subj)
        s['libre_estimated'] = s['Libre GL'] + offset
        s['glucose_primary'] = s['Dexcom GL']
        fill_mask = s['glucose_primary'].isna() & s['libre_estimated'].notna()
        s.loc[fill_mask, 'glucose_primary'] = s.loc[fill_mask, 'libre_estimated']

        method = '최근구간오프셋' if subj in TREND_SUBJECTS else '전체평균오프셋'
        s['data_quality'] = np.where(
            s['Dexcom GL'].notna(), 'Dexcom_실측',
            np.where(fill_mask, f'Libre_{method}대체', '결측')
        )
        corrected_frames.append(s)

    return pd.concat(corrected_frames, ignore_index=True)

### 8단계 최종 실행

In [ ]:
final_v2 = fill_dexcom_with_offset_libre_v2(raw)
print(final_v2['data_quality'].value_counts())

# 최종 확인 결과 (실제로 나왔던 값, 참고용):
#   Dexcom_실측              581994  (84.6%)
#   Dexcom만_신뢰불가Libre제외    53985  (7.9%)   -- 007,032,048,039
#   Libre_전체평균오프셋대체      36660  (5.3%)
#   Libre_최근구간오프셋대체      14941  (2.2%)
final_v2.to_csv('cgmacros_glucose_primary_v2.csv', index=False, encoding='utf-8-sig')

## 9. 최종 전처리 데이터셋 저장 (식사 라벨 포함)

지금까지 `raw`/`final_v2`는 혈당(Dexcom GL, Libre GL)만 가져왔다 — `load_all_subjects`가
`["subject_id", "Timestamp", "Dexcom GL", "Libre GL"]`만 골라왔기 때문이다. 각 참가자
CSV에는 `Meal Type`, `Carbs`, `Protein`, `Fat`, `Fiber`, `Calories`, `Amount Consumed` 같은
식사 라벨도 같은 파일 안에 같이 들어있으므로, 이걸 다시 읽어와서 `glucose_primary`와
합친 뒤 최종 데이터셋으로 저장한다.


In [ ]:
def load_all_subjects_with_meals(base_dir):
    """혈당뿐 아니라 식사 라벨(Meal Type, Carbs, Protein, Fat, Fiber, Calories, Amount Consumed)까지
    포함해서 전체 참가자 파일을 읽어온다. (glucose_primary와 합칠 때 쓸 원본 식사 정보)"""
    meal_cols_candidates = ['Meal Type', 'Carbs', 'Protein', 'Fat', 'Fiber', 'Calories', 'Amount Consumed']
    rows = []
    participant_dirs = sorted(glob.glob(os.path.join(base_dir, "CGMacros-*")))
    if not participant_dirs:
        participant_dirs = [base_dir]

    for pdir in participant_dirs:
        csv_files = glob.glob(os.path.join(pdir, "CGMacros-*.csv"))
        if not csv_files:
            csv_files = glob.glob(os.path.join(pdir, "*.csv"))
        for csv_path in csv_files:
            try:
                df = pd.read_csv(csv_path)
            except Exception as e:
                print(f"읽기 실패: {csv_path} ({e})")
                continue
            df.columns = [c.strip() for c in df.columns]
            if "Timestamp" not in df.columns:
                continue

            subject_id = os.path.basename(csv_path).replace(".csv", "")
            df["subject_id"] = subject_id
            df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

            present_meal_cols = [c for c in meal_cols_candidates if c in df.columns]
            keep_cols = ["subject_id", "Timestamp"] + present_meal_cols
            rows.append(df[keep_cols])

    if not rows:
        raise RuntimeError("로딩된 파일이 없습니다. CONFIG 경로를 확인하세요.")
    return pd.concat(rows, ignore_index=True)

In [ ]:
# 1) 식사 라벨 포함해서 다시 로딩
meals_raw = load_all_subjects_with_meals(CONFIG["cgmacros_dir"])
print("식사 라벨 컬럼:", [c for c in meals_raw.columns if c not in ('subject_id', 'Timestamp')])
print("전체 행:", len(meals_raw))

# 2) 최종 혈당(glucose_primary, data_quality)과 식사 라벨을 subject_id + Timestamp 기준으로 합치기
final_dataset = final_v2[['subject_id', 'Timestamp', 'Dexcom GL', 'Libre GL',
                           'glucose_primary', 'data_quality']].merge(
    meals_raw, on=['subject_id', 'Timestamp'], how='left'
)

print("\n합친 후 전체 행:", len(final_dataset))
print("식사가 기록된 행 수 (Meal Type not null):", final_dataset['Meal Type'].notna().sum())
final_dataset.head()

### 저장

In [ ]:
OUTPUT_PATH = "cgmacros_preprocessed_final.csv"
final_dataset.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_PATH}  ({len(final_dataset)}행, {final_dataset['subject_id'].nunique()}명)")

# data_quality별 분포도 같이 확인 (최종 데이터셋의 신뢰도 구성 요약)
print(final_dataset['data_quality'].value_counts())

## 다음 단계 (이 노트북 이후에 진행)

`glucose_primary` 컬럼을 기준으로:
1. 식전 baseline(끼니 직전 10분 평균), 식후 peak / delta_peak / iAUC 재계산
2. 매크로영양소·칼로리 정합성 검증 및 이상치 제거 (기존 전처리 로직 적용)
3. EDA (진단군별 반응크기, 영양소-반응 상관분석, 분산분해 등)
4. 이후 더미데이터 생성 단계로 진행


In [ ]:
import os
print(os.getcwd())

---
# 2. 끼니 단위 피처 테이블 생성 (02_meal_features)
---

# CGMacros 끼니 단위 피처 테이블 생성

`cgmacros_preprocessing_full.ipynb`에서 만든 `cgmacros_preprocessed_final.csv`
(센서 통합·보정이 끝난 `glucose_primary` + 식사 라벨)를 입력으로 받아서,
"이 끼니의 영양성분 → 식후 혈당 반응(baseline/peak/delta_peak/iAUC)"
형태의 끼니 단위 학습 데이터로 요약한다.

**작업 순서**
1. 전처리 완료 CSV 로딩
2. 매크로영양소·칼로리 정합성 검증 (이상치 제거)
3. 식사 이벤트 완전성 체크 (결측 낀 식사는 제외)
4. 식전 baseline / 식후 peak·delta_peak·iAUC 계산
5. 최종 끼니 단위 피처 테이블 저장


## 0. 라이브러리 및 설정

In [ ]:
import numpy as np
import pandas as pd

INPUT_PATH = "cgmacros_preprocessed_final.csv"   # cgmacros_preprocessing_full.ipynb의 최종 산출물

PRE_WINDOW_MIN = 30       # 식전 baseline 산출 구간 (식사 시작 -10분 ~ 0분)
POST_WINDOW_MIN = 120     # 식후 관찰 구간 (peak/iAUC 탐색 범위)
MAX_ALLOWED_GAP_MIN = 15  # 이 구간 안에 이 이상 긴 연속 결측이 있으면 해당 식사는 신뢰 불가로 제외

df = pd.read_csv(INPUT_PATH, parse_dates=["Timestamp"])
print(f"로딩 완료: {len(df)}행, 참가자 {df['subject_id'].nunique()}명")
df.head()

## 1. 매크로영양소 · 칼로리 정합성 검증

① 탄수화물×4 + 단백질×4 + 지방×9 (Atwater 계수, 식이섬유는 관례상 제외)와 기록된 칼로리를 비교해
   심하게 어긋나는 행을 이상치로 제거
② 식이섬유는 영양성분표 정의상 탄수화물의 하위 항목이라 탄수화물보다 클 수 없음 -> 위반 행 제거


In [ ]:
def flag_macro_outliers(meals, calorie_tolerance_pct=30):
    """매크로영양소로 계산한 칼로리와 기록된 칼로리가 크게 다르거나,
    식이�섬유가 탄수화물보다 많은 행을 이상치로 플래그한다."""
    m = meals.copy()
    m['calc_calories'] = m['Carbs'] * 4 + m['Protein'] * 4 + m['Fat'] * 9
    m['calorie_diff_pct'] = ((m['calc_calories'] - m['Calories']).abs() / m['Calories'].replace(0, np.nan)) * 100

    calorie_bad = m['calorie_diff_pct'] > calorie_tolerance_pct
    fiber_bad = m['Fiber'] > m['Carbs']

    m['is_outlier'] = calorie_bad.fillna(False) | fiber_bad.fillna(False)
    return m


meal_rows = df[df['Meal Type'].notna()].copy()
print(f"전체 식사 이벤트: {len(meal_rows)}건")

meal_rows = flag_macro_outliers(meal_rows)
n_outliers = meal_rows['is_outlier'].sum()
print(f"이상치로 플래그된 식사: {n_outliers}건 ({n_outliers/len(meal_rows)*100:.1f}%)")

meal_rows_clean = meal_rows[~meal_rows['is_outlier']].reset_index(drop=True)
print(f"정제 후 식사 이벤트: {len(meal_rows_clean)}건")

## 2. 식사 이벤트 완전성 체크

식전 baseline(-10~0분) ~ 식후 관찰구간(0~120분) 안에, `data_quality`가 결측인 구간이
`MAX_ALLOWED_GAP_MIN`분 넘게 연속으로 끼어있으면 그 식사는 신뢰할 수 없다고 보고 제외한다.


In [ ]:
def is_meal_window_complete(glucose_series, meal_time, pre_window_min, post_window_min, max_allowed_gap_min):
    """식전~식후 관찰구간 안에 max_allowed_gap_min분 넘는 연속 결측이 있으면 False를 반환한다."""
    window = glucose_series[
        (glucose_series.index >= meal_time - pd.Timedelta(minutes=pre_window_min)) &
        (glucose_series.index <= meal_time + pd.Timedelta(minutes=post_window_min))
    ]
    if window.empty:
        return False
    is_na = window.isna()
    if not is_na.any():
        return True
    group = (is_na != is_na.shift()).cumsum()
    max_gap = is_na.groupby(group).sum().max()
    return max_gap <= max_allowed_gap_min


# 참가자별 glucose_primary 시계열을 미리 인덱싱해두면 조회가 빠르다
glucose_by_subject = {
    subj: g.set_index('Timestamp')['glucose_primary'].sort_index()
    for subj, g in df.groupby('subject_id')
}

completeness = []
for _, row in meal_rows_clean.iterrows():
    series = glucose_by_subject[row['subject_id']]
    ok = is_meal_window_complete(series, row['Timestamp'], PRE_WINDOW_MIN, POST_WINDOW_MIN, MAX_ALLOWED_GAP_MIN)
    completeness.append(ok)

meal_rows_clean['window_complete'] = completeness
n_incomplete = (~meal_rows_clean['window_complete']).sum()
print(f"결측이 껴서 신뢰 불가로 제외되는 식사: {n_incomplete}건 ({n_incomplete/len(meal_rows_clean)*100:.1f}%)")

meal_rows_final = meal_rows_clean[meal_rows_clean['window_complete']].reset_index(drop=True)
print(f"최종 사용 가능한 식사 이벤트: {len(meal_rows_final)}건")

## 3. 식전 baseline / 식후 peak · delta_peak · iAUC 계산

- **baseline**: 식사 -10분 ~ 0분 구간의 `glucose_primary` 평균
- **peak**: 식사 0분 ~ 120분 구간의 `glucose_primary` 최댓값 (절대값)
- **delta_peak**: peak - baseline (상승폭)
- **iAUC**: baseline보다 위에 있는 부분만 사다리꼴 적분으로 계산한 곡선하면적
- **data_quality_in_window**: 이 식사의 계산 구간 안에 실측이 아닌 값(오프셋대체/보간)이 섞였는지 표시
  (모델링 시 "완전 실측 구간만 쓴 결과"와 비교해볼 수 있도록 남겨둠)


In [ ]:
def compute_iauc(window_series, baseline):
    """baseline보다 위에 있는 부분만 사다리꼴 적분으로 곡선하면적을 계산한다 (분 단위 시간축)."""
    t = (window_series.index - window_series.index[0]).total_seconds() / 60  # 분 단위 경과시간
    values_above = (window_series.values - baseline).clip(min=0)
    return float(np.trapz(values_above, t))


def compute_meal_features(meal_rows, glucose_by_subject, quality_by_subject,
                           pre_window_min=PRE_WINDOW_MIN, post_window_min=POST_WINDOW_MIN):
    records = []
    for _, row in meal_rows.iterrows():
        series = glucose_by_subject[row['subject_id']]
        quality = quality_by_subject[row['subject_id']]
        t0 = row['Timestamp']

        pre_win = series[(series.index >= t0 - pd.Timedelta(minutes=pre_window_min)) & (series.index <= t0)]
        baseline = pre_win.mean()

        post_win = series[(series.index >= t0) & (series.index <= t0 + pd.Timedelta(minutes=post_window_min))]
        post_win = post_win.dropna()
        if post_win.empty or pd.isna(baseline):
            continue

        peak = post_win.max()
        delta_peak = peak - baseline
        iauc = compute_iauc(post_win, baseline)

        full_win_idx = series[(series.index >= t0 - pd.Timedelta(minutes=pre_window_min)) &
                               (series.index <= t0 + pd.Timedelta(minutes=post_window_min))].index
        q_win = quality.reindex(full_win_idx)
        non_real_ratio = (q_win != 'Dexcom_실측').mean() if len(q_win) else np.nan

        records.append({
            'subject_id': row['subject_id'],
            'meal_time': t0,
            'meal_type': row['Meal Type'],
            'carbs_g': row['Carbs'],
            'protein_g': row['Protein'],
            'fat_g': row['Fat'],
            'fiber_g': row['Fiber'],
            'calories': row['Calories'],
            'baseline': round(baseline, 1),
            'peak': round(peak, 1),
            'delta_peak': round(delta_peak, 1),
            'iauc': round(iauc, 1),
            'non_real_data_ratio': round(non_real_ratio, 3) if pd.notna(non_real_ratio) else np.nan,
        })

    return pd.DataFrame(records)


quality_by_subject = {
    subj: g.set_index('Timestamp')['data_quality'].sort_index()
    for subj, g in df.groupby('subject_id')
}

meal_features = compute_meal_features(meal_rows_final, glucose_by_subject, quality_by_subject)
print(f"최종 끼니 단위 피처 테이블: {len(meal_features)}행")
meal_features.head()

### 실측 비율 확인 — 대체값이 섞인 식사가 얼마나 되는지

In [ ]:
fully_real = (meal_features['non_real_data_ratio'] == 0).sum()
print(f"계산 구간 전체가 Dexcom 실측인 식사: {fully_real}건 ({fully_real/len(meal_features)*100:.1f}%)")
print(f"대체값이 조금이라도 섞인 식사: {len(meal_features) - fully_real}건")
print(meal_features['non_real_data_ratio'].describe())

## 4. 저장

In [ ]:
OUTPUT_PATH = "cgmacros_meal_features.csv"
meal_features.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_PATH}  ({len(meal_features)}행, {meal_features['subject_id'].nunique()}명)")

meal_features.groupby('subject_id')[['baseline', 'delta_peak', 'iauc']].describe().head()

## 다음 단계

이 `cgmacros_meal_features.csv`를 기준으로:
1. EDA (진단군별 반응크기, 영양소-반응 상관분석, 분산분해 등 — 이전 노트북에서 했던 것과 동일한 분석)
2. 영양소-반응 관계식(회귀 계수) 추출
3. 더미데이터 생성


---
# 3. 탐색적 데이터 분석 (03_eda)
---

# CGMacros 식후 혈당 반응 EDA (전처리 개선판)

`cgmacros_meal_features.csv`(센서 통합·트리밍·완전성 체크까지 끝난 끼니 단위 피처)를 기반으로,
- 진단군별 반응 크기 차이
- 영양소-반응 상관분석
- 분산 분해(사람간 vs 끼니별)
- 모델 비교(선형회귀/랜덤포레스트/XGBoost)
- 임상지표·식전혈당 추가 효과
- 개인별(LOO) 예측 가능성

을 확인한다. 이전에 CGMacros 원본으로 했던 분석과 같은 구조이며, 이번엔 개선된 `glucose_primary`
(Dexcom/Libre 통합·보정·트리밍 완료) 기준으로 재수행한다.


## 0. 라이브러리 · 데이터 로딩

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost 미설치 -- pip install xgboost 후 다시 실행하면 XGBoost 결과도 포함됩니다.")

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

CONFIG = {
    "meal_features_path": "cgmacros_meal_features.csv",
    "cgmacros_dir": r"C:\Users\smhrd1\Desktop\데이터셋 혈당\CGMacros",  # bio.csv 등 임상변수 로딩용
}

meals = pd.read_csv(CONFIG["meal_features_path"], parse_dates=["meal_time"])
print(f"끼니 데이터: {len(meals)}행, 참가자 {meals['subject_id'].nunique()}명")
meals.head()

### bio.csv(임상변수·진단군) 로딩 및 병합

CGMacros 배포본에는 참가자별 인적사항/임상변수가 담긴 `bio.csv`가 별도로 있다. 여기서
진단군(건강/전당뇨/2형당뇨), BMI, A1c, 공복혈당, 나이를 가져와 `meals`에 합친다.

In [ ]:
def load_bio(cgmacros_dir):
    candidates = glob.glob(os.path.join(cgmacros_dir, "bio.csv")) + \
                 glob.glob(os.path.join(cgmacros_dir, "**", "bio.csv"), recursive=True)
    if not candidates:
        raise FileNotFoundError("bio.csv를 찾지 못했습니다. CONFIG['cgmacros_dir'] 경로를 확인하세요.")
    bio = pd.read_csv(candidates[0])
    bio.columns = [c.strip() for c in bio.columns]
    return bio


bio = load_bio(CONFIG["cgmacros_dir"])
print("bio.csv 컬럼:", list(bio.columns))

# subject_id 형식 맞추기 (bio.csv의 참가자 식별자가 CGMacros-0XX 형식이 아닐 수 있어 확인 필요)
id_col = [c for c in bio.columns if 'subject' in c.lower() or 'id' in c.lower()][0]
print("식별자 컬럼으로 추정:", id_col)
bio.head()

In [ ]:
# subject_id 포맷을 'CGMacros-0XX'로 맞춰서 병합 (id_col 값이 숫자만 있는 경우 대비)
def normalize_subject_id(x):
    x = str(x).strip()
    if x.startswith('CGMacros'):
        return x
    return f"CGMacros-{int(x):03d}"

bio['subject_id'] = bio[id_col].apply(normalize_subject_id)

clinical_cols_candidates = ['Gender', 'Age', 'BMI', 'A1c PDL (Lab)', 'Fasting GLU - PDL (Lab)']
clinical_cols = [c for c in clinical_cols_candidates if c in bio.columns]
print("사용할 임상변수:", clinical_cols)

meals = meals.merge(bio[['subject_id'] + clinical_cols], on='subject_id', how='left')

# 진단군 라벨링 -- HbA1c 기준 (원본 CGMacros 데이터 설명 그대로: <5.7 건강군 / 5.7~6.4 전당뇨 / >6.4 2형당뇨)
# 04_risk_model.ipynb에서 이 기준으로 만든 라벨이 실제 참가자 라벨과 100% 일치함을 검증했음
# (참고: 이전에는 공복혈당 100/126 기준 자동분류를 썼으나, 04번과 기준이 달라 통일함)
a1c_col = 'A1c PDL (Lab)'

def classify_by_a1c(a1c):
    if pd.isna(a1c):
        return None
    elif a1c < 5.7:
        return '건강군'
    elif a1c <= 6.4:
        return '전당뇨'
    else:
        return '2형당뇨'

meals['group'] = meals[a1c_col].apply(classify_by_a1c)
print(meals.groupby('subject_id')['group'].first().value_counts())
meals.head()


## 1. 진단군별 식후 혈당 반응 크기 비교

건강군 < 전당뇨 < 2형당뇨 순으로 반응 크기(delta_peak, iauc)가 커지는지 확인한다.

In [ ]:
group_summary = meals.groupby('group')[['delta_peak', 'iauc']].agg(['mean', 'std', 'count'])
print(group_summary)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
meals.boxplot(column='delta_peak', by='group', ax=axes[0])
axes[0].set_title('진단군별 Δpeak'); axes[0].set_xlabel(''); plt.sca(axes[0])
meals.boxplot(column='iauc', by='group', ax=axes[1])
axes[1].set_title('진단군별 iAUC'); axes[1].set_xlabel('')
plt.suptitle('')
plt.tight_layout()
plt.savefig('01_그룹별_반응크기.png', dpi=150)
plt.show()

## 2. 영양소-반응 상관분석

탄수화물/단백질/지방/식이섬유/칼로리와 iAUC·Δpeak 간의 피어슨 상관계수를 계산한다.

In [ ]:
NUTRIENTS = ['carbs_g', 'protein_g', 'fat_g', 'fiber_g', 'calories']

corr_rows = []
for nut in NUTRIENTS:
    valid = meals[[nut, 'iauc', 'delta_peak']].dropna()
    r_iauc, p_iauc = pearsonr(valid[nut], valid['iauc'])
    r_delta, p_delta = pearsonr(valid[nut], valid['delta_peak'])
    corr_rows.append({'nutrient': nut, 'pearson_r_iauc': round(r_iauc, 3), 'p_iauc': round(p_iauc, 4),
                       'pearson_r_delta_peak': round(r_delta, 3), 'p_delta_peak': round(p_delta, 4)})

corr_df = pd.DataFrame(corr_rows)
print(corr_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(NUTRIENTS))
ax.bar(x - 0.2, corr_df['pearson_r_iauc'], width=0.4, label='iAUC')
ax.bar(x + 0.2, corr_df['pearson_r_delta_peak'], width=0.4, label='Δpeak')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(NUTRIENTS, rotation=20)
ax.set_ylabel('Pearson r'); ax.legend(); ax.set_title('영양소-반응 상관계수')
plt.tight_layout()
plt.savefig('02_영양소_상관분석.png', dpi=150)
plt.show()

### 산점도 — 탄수화물과 반응 (그룹별 회귀선 비교)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for grp, g in meals.dropna(subset=['group']).groupby('group'):
    ax.scatter(g['carbs_g'], g['iauc'], alpha=0.4, s=15, label=grp)
    if len(g) > 5:
        coef = np.polyfit(g['carbs_g'], g['iauc'], 1)
        xs = np.linspace(g['carbs_g'].min(), g['carbs_g'].max(), 50)
        ax.plot(xs, np.polyval(coef, xs), linewidth=2)
ax.set_xlabel('탄수화물(g)'); ax.set_ylabel('iAUC')
ax.set_title('탄수화물-iAUC (그룹별 회귀선)')
ax.legend()
plt.tight_layout()
plt.savefig('03_탄수화물_산점도.png', dpi=150)
plt.show()

## 3. 예측 모델 비교 — 영양소만으로 새로운 사람을 예측할 수 있는가

GroupKFold(subject 기준)로, 같은 사람의 다른 끼니가 train/test에 섞이지 않도록 분리해서 검증한다.

In [ ]:
def cv_eval(X, y, groups, model_fn, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    preds = np.zeros_like(y, dtype=float)
    for train_idx, test_idx in gkf.split(X, y, groups):
        model = model_fn()
        model.fit(X[train_idx], y[train_idx])
        preds[test_idx] = model.predict(X[test_idx])
    return r2_score(y, preds), preds


meals_valid = meals.dropna(subset=NUTRIENTS + ['iauc', 'delta_peak']).reset_index(drop=True)
X_nutrients = meals_valid[NUTRIENTS].values
y_iauc = meals_valid['iauc'].values
y_delta = meals_valid['delta_peak'].values
groups_all = meals_valid['subject_id'].values

models = {
    '탄수화물만(선형)': (lambda: LinearRegression(), meals_valid[['carbs_g']].values),
    '전체영양소(선형)': (lambda: LinearRegression(), X_nutrients),
    '전체영양소(랜덤포레스트)': (lambda: RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42), X_nutrients),
}
if HAS_XGB:
    models['전체영양소(XGBoost)'] = (
        lambda: XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, random_state=42),
        X_nutrients
    )

results = []
for name, (model_fn, X) in models.items():
    r2_iauc, _ = cv_eval(X, y_iauc, groups_all, model_fn)
    r2_delta, _ = cv_eval(X, y_delta, groups_all, model_fn)
    results.append({'모델': name, 'iAUC_R2': round(r2_iauc, 3), 'Δpeak_R2': round(r2_delta, 3)})

model_comparison = pd.DataFrame(results)
print(model_comparison.to_string(index=False))
model_comparison.to_csv('model_comparison_table.csv', index=False, encoding='utf-8-sig')

## 4. 진단군 내부에서는 더 잘 예측될까?

In [ ]:
per_group_results = []
for grp, g in meals_valid.dropna(subset=['group']).groupby('group'):
    if len(g) < 20 or g['subject_id'].nunique() < 3:
        continue
    Xg = g[NUTRIENTS].values
    r2_iauc, _ = cv_eval(Xg, g['iauc'].values, g['subject_id'].values, lambda: LinearRegression(),
                          n_splits=min(5, g['subject_id'].nunique()))
    r2_delta, _ = cv_eval(Xg, g['delta_peak'].values, g['subject_id'].values, lambda: LinearRegression(),
                           n_splits=min(5, g['subject_id'].nunique()))
    per_group_results.append({'진단군': grp, 'n_meals': len(g), 'iAUC_R2': round(r2_iauc, 3), 'Δpeak_R2': round(r2_delta, 3)})

per_group_df = pd.DataFrame(per_group_results)
print(per_group_df.to_string(index=False))

## 5. 분산 분해 — 사람 간 차이 vs 끼니별 차이

iAUC/Δpeak의 전체 분산 중 얼마가 "이 사람이 누구인가"(between-subject)에서 오고,
얼마가 "끼니마다 다름"(within-subject)에서 오는지 확인한다.

In [ ]:
def variance_decomposition(df, col):
    subject_means = df.groupby('subject_id')[col].transform('mean')
    total_var = df[col].var()
    between_var = subject_means.var()
    within_var = (df[col] - subject_means).var()
    return {
        'target': col,
        'between_subject_pct': round(between_var / total_var * 100, 1),
        'within_subject_pct': round(within_var / total_var * 100, 1),
    }

var_results = [variance_decomposition(meals_valid, 'iauc'), variance_decomposition(meals_valid, 'delta_peak')]
var_df = pd.DataFrame(var_results)
print(var_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(var_df['target'], var_df['between_subject_pct'], label='사람 간 차이')
ax.bar(var_df['target'], var_df['within_subject_pct'], bottom=var_df['between_subject_pct'], label='끼니별 차이')
ax.set_ylabel('%'); ax.legend(); ax.set_title('분산 분해')
plt.tight_layout()
plt.savefig('05_분산분해.png', dpi=150)
plt.show()

## 6. 임상지표를 더하면 예측력이 좋아질까?

영양소만 / 임상지표만 / 영양소+임상지표 세 조합을 비교한다.

In [ ]:
clin_cols = [c for c in clinical_cols if c != 'Gender']
meals_clin = meals.dropna(subset=NUTRIENTS + clin_cols + ['iauc', 'delta_peak']).reset_index(drop=True)

if len(meals_clin) > 30:
    X_nut = meals_clin[NUTRIENTS].values
    X_clin = meals_clin[clin_cols].values
    X_both = meals_clin[NUTRIENTS + clin_cols].values
    y_iauc_c = meals_clin['iauc'].values
    y_delta_c = meals_clin['delta_peak'].values
    groups_c = meals_clin['subject_id'].values

    clin_results = []
    for name, X in [('영양소만', X_nut), ('임상지표만', X_clin), ('영양소+임상지표', X_both)]:
        r2_i, _ = cv_eval(X, y_iauc_c, groups_c, lambda: LinearRegression())
        r2_d, _ = cv_eval(X, y_delta_c, groups_c, lambda: LinearRegression())
        clin_results.append({'입력': name, 'iAUC_R2': round(r2_i, 3), 'Δpeak_R2': round(r2_d, 3)})

    clin_df = pd.DataFrame(clin_results)
    print(clin_df.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
    axes[0].bar(clin_df['입력'], clin_df['iAUC_R2']); axes[0].set_title('iAUC R²'); axes[0].tick_params(axis='x', rotation=20)
    axes[1].bar(clin_df['입력'], clin_df['Δpeak_R2']); axes[1].set_title('Δpeak R²'); axes[1].tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.savefig('06_임상지표_모델비교.png', dpi=150)
    plt.show()
else:
    print("임상변수 결측이 많아 비교 표본이 부족합니다. bio.csv 병합 결과를 확인하세요.")

## 7. 식전 혈당(baseline)을 추가하면?

영양소에 baseline을 추가해서 iAUC/Δpeak/peak(절대값) 예측력이 어떻게 바뀌는지 확인한다.

In [ ]:
meals_bl = meals_valid.dropna(subset=['baseline', 'peak']).reset_index(drop=True)
X_nut2 = meals_bl[NUTRIENTS].values
X_nut_bl = meals_bl[NUTRIENTS + ['baseline']].values
y_iauc2 = meals_bl['iauc'].values
y_delta2 = meals_bl['delta_peak'].values
y_peak2 = meals_bl['peak'].values
groups2 = meals_bl['subject_id'].values

def make_xgb():
    if HAS_XGB:
        return XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, random_state=42)
    return RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42)

baseline_results = []
for label, X in [('영양소만', X_nut2), ('영양소+식전혈당', X_nut_bl)]:
    r2_i, _ = cv_eval(X, y_iauc2, groups2, make_xgb)
    r2_d, _ = cv_eval(X, y_delta2, groups2, make_xgb)
    r2_p, _ = cv_eval(X, y_peak2, groups2, make_xgb)
    baseline_results.append({'입력': label, 'iAUC_R2': round(r2_i, 3), 'Δpeak_R2': round(r2_d, 3), 'peak_R2': round(r2_p, 3)})

baseline_df = pd.DataFrame(baseline_results)
print(baseline_df.to_string(index=False))

# 참고: baseline 하나만으로 peak를 예측하면 어느 정도인지 (자기상관 기여도 확인용)
r2_peak_baseline_only, _ = cv_eval(meals_bl[['baseline']].values, y_peak2, groups2, make_xgb)
print(f"\nbaseline 단독으로 peak 예측 R² = {r2_peak_baseline_only:.3f}  (영양소+baseline 대비 baseline 단독 기여도 확인용)")

## 8. 개인별로 보정하면? — 각자 자신의 데이터로 leave-one-out 예측

각 사람의 데이터가 충분한 경우(예: 10건 이상), 그 사람의 데이터만으로 leave-one-out
교차검증을 해서 "자기 자신의 과거 데이터로 예측 가능한지"를 확인한다.

In [ ]:
from sklearn.model_selection import LeaveOneOut

loo_results = []
for subj, g in meals_valid.groupby('subject_id'):
    if len(g) < 10:
        continue
    X = g[NUTRIENTS].values
    y = g['iauc'].values
    loo = LeaveOneOut()
    preds = np.zeros_like(y, dtype=float)
    for train_idx, test_idx in loo.split(X):
        model = LinearRegression()
        model.fit(X[train_idx], y[train_idx])
        preds[test_idx] = model.predict(X[test_idx])
    r2 = r2_score(y, preds)
    loo_results.append({'subject_id': subj, 'n_meals': len(g), 'loo_r2': round(r2, 3)})

loo_df = pd.DataFrame(loo_results).sort_values('loo_r2', ascending=False)
print(loo_df.to_string(index=False))

pct_positive = (loo_df['loo_r2'] > 0).mean() * 100
print(f"\n개인별 LOO R²>0인 비율: {pct_positive:.1f}%")

fig, ax = plt.subplots(figsize=(11, 4))
colors = meals_valid.groupby('subject_id')['group'].first().reindex(loo_df['subject_id'])
ax.bar(loo_df['subject_id'], loo_df['loo_r2'])
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('개인별 LOO R²'); ax.set_title('개인별(Leave-One-Out) 예측 가능성')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig('07_개인별_LOO_R2.png', dpi=150)
plt.show()

## 결론 요약

- 진단군에 따라 반응 크기 자체가 다른지 (1번)
- 영양소가 반응과 어떤 상관을 보이는지, 개인/그룹별로 다른지 (2번)
- 영양소만으로 새로운 사람을 예측할 수 있는 수준인지 (3, 4번)
- 반응의 분산이 개인차 vs 끼니차 중 어디서 더 크게 오는지 (5번)
- 임상지표·식전혈당을 추가하면 얼마나 개선되는지, peak(절대값)는 baseline 자기상관에
  얼마나 좌우되는지 (6, 7번)
- 개인별 자기 데이터로는 얼마나 예측 가능한지 (8번)



---
# 4. 콜드스타트 통합 모델 (04_risk_model)
---

In [ ]:
# namespace reset
for _v in list(dir()):
    if not _v.startswith("_") and _v not in ("In","Out","get_ipython","exit","quit","open"):
        try: exec(f"del {_v}")
        except: pass
import gc; gc.collect()
print("namespace reset done")

# 04. 식후 혈당 예측 모델

지금까지의 실험(EDA, baseline 추가, 진단군 추가 등)에서 확정된 피처셋으로,
**여러 모델을 새로 학습·비교**하고, 최종 모델로 **식전혈당 대비 예상 상승분**을
계산한다.

**팀 결정 반영 사항**
- GI(혈당지수) 보정 로직은 사용하지 않는다 (근거 부족한 휴리스틱으로 판단)
- 위험도 등급(낮음/보통/위험) 분류도 사용하지 않는다
- 최종 챗봇 출력은 "식전혈당 + 예상 상승분" 형태로 통일한다
  (예: "지금 145에서 약 30 정도 올라서 175 근처일 것 같아요")

**최종 피처셋(6개)**: 탄수화물, 단백질, 지방, 식이섬유, baseline(식전혈당), 진단군(원-핫)
**타깃**: peak (baseline + 식후 상승분) — 예측 후 상승분(peak-baseline)은 그 자리에서 계산

## 0. 라이브러리 · 데이터 로딩

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, confusion_matrix, classification_report

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost 미설치 -- pip install xgboost 후 재실행하면 XGBoost 결과도 포함됩니다.")

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("lightgbm 미설치 -- pip install lightgbm 후 재실행하면 LightGBM 결과도 포함됩니다.")

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

CONFIG = {
    "meal_features_path": r"C:\Users\smhrd1\Desktop\CGMacros\cgmacros_meal_features.csv",
    "cgmacros_dir": r"C:\Users\smhrd1\Desktop\CGMacros",
    "korean_food_path": r"C:\Users\smhrd1\Desktop\CGMacros\한식_영양성분_427건.csv",
}

NUTRIENTS = ['carbs_g', 'protein_g', 'fat_g', 'fiber_g']

meals = pd.read_csv(CONFIG["meal_features_path"], parse_dates=["meal_time"])
print(f"끼니 데이터: {len(meals)}행, 참가자 {meals['subject_id'].nunique()}명")

### bio.csv 병합 (진단군 라벨)

In [ ]:
def load_bio(cgmacros_dir):
    candidates = glob.glob(os.path.join(cgmacros_dir, "bio.csv")) + \
                 glob.glob(os.path.join(cgmacros_dir, "**", "bio.csv"), recursive=True)
    if not candidates:
        raise FileNotFoundError("bio.csv를 찾지 못했습니다. CONFIG['cgmacros_dir'] 경로를 확인하세요.")
    bio = pd.read_csv(candidates[0])
    bio.columns = [c.strip() for c in bio.columns]
    return bio

def normalize_subject_id(x):
    x = str(x).strip()
    if x.startswith('CGMacros'):
        return x
    return f"CGMacros-{int(x):03d}"

bio = load_bio(CONFIG["cgmacros_dir"])
id_col = [c for c in bio.columns if 'subject' in c.lower() or 'id' in c.lower()][0]
bio['subject_id'] = bio[id_col].apply(normalize_subject_id)

# 진단군 라벨링 -- HbA1c 기준 (03_eda.ipynb와 동일. 실제 참가자 라벨과 100% 일치 검증됨)
a1c_col = 'A1c PDL (Lab)'
meals = meals.merge(bio[['subject_id', a1c_col]], on='subject_id', how='left')

def classify_by_a1c(a1c):
    if pd.isna(a1c):
        return None
    elif a1c < 5.7:
        return '건강군'
    elif a1c <= 6.4:
        return '전당뇨'
    else:
        return '2형당뇨'

meals['group'] = meals[a1c_col].apply(classify_by_a1c)
print(meals.groupby('subject_id')['group'].first().value_counts())


## 1. 최종 피처셋 구성

In [ ]:
meals_final = meals.dropna(subset=NUTRIENTS + ['baseline', 'peak', 'group']).reset_index(drop=True)

group_dummies = pd.get_dummies(meals_final['group'], prefix='group')
FEATURE_COLS = NUTRIENTS + ['baseline'] + list(group_dummies.columns)

X = pd.concat([meals_final[NUTRIENTS + ['baseline']], group_dummies], axis=1).values
y = meals_final['peak'].values
groups = meals_final['subject_id'].values

# 모델을 신뢰할 수 있는 baseline 상한 (이 값을 넘는 입력은 학습 데이터가 희박해 예측이 불안정함)
MAX_RELIABLE_BASELINE = meals_final['baseline'].quantile(0.90)

print(f"최종 학습 데이터: {len(meals_final)}건, 피처 {len(FEATURE_COLS)}개")
print("피처 목록:", FEATURE_COLS)
print(f"신뢰 가능한 baseline 상한(90th pct): {MAX_RELIABLE_BASELINE:.1f}")

## 2. 여러 모델 비교 (새로 학습·검증)

GroupKFold(subject 기준)로 동일한 조건에서 여러 모델을 비교한다.

In [ ]:
def cv_eval_full(X, y, groups, model_fn, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    preds = np.zeros_like(y, dtype=float)
    for train_idx, test_idx in gkf.split(X, y, groups):
        model = model_fn()
        model.fit(X[train_idx], y[train_idx])
        preds[test_idx] = model.predict(X[test_idx])
    r2 = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)
    mape = np.mean(np.abs((y - preds) / y)) * 100
    return r2, mae, mape, preds


candidate_models = {
    '선형회귀': lambda: LinearRegression(),
    'Ridge': lambda: Ridge(alpha=10.0),
    '랜덤포레스트': lambda: RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42),
    'GradientBoosting(sklearn)': lambda: GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42),
}
if HAS_XGB:
    candidate_models['XGBoost'] = lambda: XGBRegressor(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42
    )
if HAS_LGBM:
    candidate_models['LightGBM'] = lambda: LGBMRegressor(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=-1
    )

model_results = []
model_preds = {}
for name, model_fn in candidate_models.items():
    r2, mae, mape, preds = cv_eval_full(X, y, groups, model_fn)
    model_results.append({'모델': name, 'R2': round(r2, 3), 'MAE(mg/dL)': round(mae, 1), 'MAPE(%)': round(mape, 1)})
    model_preds[name] = preds
    print(f"{name:30s}  R²={r2:.3f}  MAE={mae:.1f}  MAPE={mape:.1f}%")

model_comparison_df = pd.DataFrame(model_results).sort_values('R2', ascending=False)

PREFER = 'LightGBM'
top_r2 = model_comparison_df.iloc[0]['R2']
prefer_row = model_comparison_df[model_comparison_df['모델'] == PREFER]
if len(prefer_row) and abs(top_r2 - prefer_row.iloc[0]['R2']) < 0.005:
    idx = prefer_row.index[0]
    model_comparison_df = pd.concat([
        model_comparison_df.loc[[idx]],
        model_comparison_df.drop(idx)
    ])
    print(f"R² 차이 < 0.005 -> 추론 속도 우선으로 {PREFER} 선정")

print()
print(model_comparison_df.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(model_comparison_df['모델'], model_comparison_df['R2'])
ax.set_xlabel('GroupKFold R² (peak 예측)')
ax.set_title('모델별 성능 비교 (새로 학습)')
plt.tight_layout()
plt.savefig('04_모델비교.png', dpi=150)
plt.show()

## 3. 최종 모델 선정 및 상승분 예측 성능 확인

R²가 가장 높은 모델을 최종 모델로 선정하고, "식전혈당 대비 얼마나 오르는지"(상승분)를
얼마나 잘 맞히는지 MAE(평균절대오차)로 같이 확인한다. (등급 분류는 사용하지 않음)

In [ ]:
from sklearn.metrics import mean_absolute_error

BEST_MODEL_NAME = model_comparison_df.iloc[0]['모델']
print(f"최종 선정 모델: {BEST_MODEL_NAME}")

gkf = GroupKFold(n_splits=5)
baseline_arr = meals_final['baseline'].values

preds = np.zeros_like(y, dtype=float)
model_fn = lambda: candidate_models[BEST_MODEL_NAME]()
for train_idx, test_idx in gkf.split(X, y, groups):
    m = model_fn()
    m.fit(X[train_idx], y[train_idx])
    preds[test_idx] = m.predict(X[test_idx])

true_delta = y - baseline_arr
pred_delta = preds - baseline_arr

# 핵심 지표
peak_r2 = r2_score(y, preds)
peak_mae = mean_absolute_error(y, preds)
peak_mape = np.mean(np.abs((y - preds) / y)) * 100
peak_precision = 100 - peak_mape
delta_mae = mean_absolute_error(true_delta, pred_delta)

print(f"peak 예측 R²: {peak_r2:.3f}  (내부 참고용)")
print(f"peak 예측 MAE: {peak_mae:.1f} mg/dL")
print(f"peak 예측 MAPE: {peak_mape:.1f}%")
print(f"peak 예측 Precision Mean: {peak_precision:.1f}%")
print(f"상승분(peak-baseline) 예측 MAE: {delta_mae:.1f} mg/dL")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(true_delta, pred_delta, alpha=0.3, s=15)
lims = [min(true_delta.min(), pred_delta.min()), max(true_delta.max(), pred_delta.max())]
ax.plot(lims, lims, color='red', linestyle='--', label='완벽히 맞춘 경우')
ax.set_xlabel('실제 상승분'); ax.set_ylabel('예측 상승분')
ax.set_title(f'상승분 예측 산점도 ({BEST_MODEL_NAME})')
ax.legend()
plt.tight_layout()
plt.savefig('04_상승분_산점도.png', dpi=150)
plt.show()


## 4. 최종 모델로 전체 데이터 재학습 (실서비스 배포용)

교차검증은 성능 확인용이었고, 실제 서비스에 쓸 모델은 **가진 데이터 전체**로 다시 학습한다.

In [ ]:
final_model = candidate_models[BEST_MODEL_NAME]()
final_model.fit(X, y)
print(f"{BEST_MODEL_NAME} 최종 모델 학습 완료 (전체 {len(meals_final)}건 사용)")

import joblib
joblib.dump(final_model, 'final_risk_model.pkl')
joblib.dump(list(group_dummies.columns), 'final_model_group_columns.pkl')
print("저장 완료: final_risk_model.pkl")

---
# 5. 개인화 모델 (train_personal_models)
---

In [ ]:
# namespace reset
for _v in list(dir()):
    if not _v.startswith("_") and _v not in ("In","Out","get_ipython","exit","quit","open"):
        try: exec(f"del {_v}")
        except: pass
import gc; gc.collect()
print("namespace reset done")

# 개인화 모델 학습

콜드스타트 모델(04_risk_model.ipynb)은 전체 참가자 데이터로 학습한 통합 모델이다.
사용자별 데이터가 쌓이면 **개인화 모델로 전환**해서 예측 정확도를 높이는 것이 목표다.

**작업 순서**
1. 데이터 로딩 및 참가자별 분할
2. 모델 비교 (Ridge / XGBoost / LightGBM) → 단일 모델 선정
3. 개인별 80/20 학습·검증
4. 콜드스타트 모델 대비 개인화 이득 비교
5. 전환 기준 분석 (끼니 수 / 진단군별)
6. 개인별 모델 저장

**피처**: 탄수화물, 단백질, 지방, 식이섬유, baseline (진단군은 개인 내 상수이므로 제외)
**타깃**: peak
**평가**: MAE (mg/dL), MAPE (%), Precision Mean (%) — 80/20 split

In [ ]:
import os
import json
import glob
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

CONFIG = {
    "meal_features_path": r"C:\Users\smhrd1\Desktop\CGMacros\cgmacros_meal_features.csv",
    "cgmacros_dir": r"C:\Users\smhrd1\Desktop\CGMacros",
    "coldstart_model_path": r"C:\Users\smhrd1\Desktop\CGMacros\final_risk_model.pkl",
}

NUTRIENTS = ['carbs_g', 'protein_g', 'fat_g', 'fiber_g']
PERSONAL_FEATURES = NUTRIENTS + ['baseline']
TARGET = 'peak'
MIN_MEALS = 10

MODEL_DIR = os.path.join(CONFIG["cgmacros_dir"], "personal_models")
os.makedirs(MODEL_DIR, exist_ok=True)

print("설정 완료")

## 1. 데이터 로딩

In [ ]:
bio = pd.read_csv(os.path.join(CONFIG["cgmacros_dir"], "bio.csv"))
bio.columns = [c.strip() for c in bio.columns]
bio['subject_id'] = bio['subject'].apply(
    lambda x: f"CGMacros-{int(str(x).strip()):03d}" if not str(x).startswith('CGMacros') else str(x).strip()
)

a1c_col = 'A1c PDL (Lab)'
meals = pd.read_csv(CONFIG["meal_features_path"], parse_dates=["meal_time"])
meals = meals.merge(bio[['subject_id', a1c_col]], on='subject_id', how='left')
meals['group'] = meals[a1c_col].apply(
    lambda a: '건강군' if a < 5.7 else ('전당뇨' if a <= 6.4 else '2형당뇨') if pd.notna(a) else None
)

meals_valid = meals.dropna(subset=PERSONAL_FEATURES + [TARGET, 'group']).reset_index(drop=True)
print(f"유효 끼니: {len(meals_valid)}건, 참가자 {meals_valid['subject_id'].nunique()}명")

meal_counts = meals_valid.groupby('subject_id').size().sort_values(ascending=False)
eligible = meal_counts[meal_counts >= MIN_MEALS]
print(f"개인화 가능 ({MIN_MEALS}끼니 이상): {len(eligible)}명")
print(f"끼니 수 범위: {eligible.min()} ~ {eligible.max()}")

## 2. 모델 정의

In [ ]:
def make_lgbm(n):
    return LGBMRegressor(
        learning_rate=0.03,
        n_estimators=120,
        num_leaves=4,
        max_depth=3,
        min_child_samples=max(5, int(n * 0.15)),
        min_split_gain=0.01,
        subsample=0.8, subsample_freq=1,
        colsample_bytree=0.7,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42, verbose=-1,
    )

def make_xgb(n):
    return XGBRegressor(
        learning_rate=0.03,
        n_estimators=120,
        max_depth=3,
        min_child_weight=max(5, int(n * 0.15)),
        gamma=0.01,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
    )

def make_ridge():
    return Ridge(alpha=10.0)


def eval_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
    return mae, mape, preds

## 3. 모델 비교 및 선정 (80/20 split)

3개 모델(Ridge / XGBoost / LightGBM)의 평균 MAPE를 비교하고, 전체 참가자에서 가장 좋은 **단일 모델**을 선정한다.

In [ ]:
all_results = []

for subj in eligible.index:
    g = meals_valid[meals_valid['subject_id'] == subj]
    X = g[PERSONAL_FEATURES].values
    y = g[TARGET].values
    n = len(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    row = {'subject_id': subj, 'n_meals': n, 'n_test': len(y_test),
           'group': g['group'].iloc[0]}

    for name, m in [('Ridge', make_ridge()),
                    ('XGBoost', make_xgb(n)),
                    ('LightGBM', make_lgbm(n))]:
        mae, mape, _ = eval_model(m, X_train, y_train, X_test, y_test)
        row[f'{name}_MAE'] = round(mae, 1)
        row[f'{name}_MAPE'] = round(mape, 1)
        row[f'{name}_Precision'] = round(100 - mape, 1)

    all_results.append(row)

results_df = pd.DataFrame(all_results)

print("=== 모델별 평균 성능 ===")
model_names = ['Ridge', 'XGBoost', 'LightGBM']
for name in model_names:
    avg_mae = results_df[f'{name}_MAE'].mean()
    avg_mape = results_df[f'{name}_MAPE'].mean()
    avg_prec = results_df[f'{name}_Precision'].mean()
    print(f"{name:10s}  MAE={avg_mae:5.1f} mg/dL   MAPE={avg_mape:5.1f}%   Precision Mean={avg_prec:5.1f}%")

BEST_MODEL = min(model_names, key=lambda k: results_df[f'{k}_MAPE'].mean())
print(f"\n>>> 선정 모델: {BEST_MODEL}")

personal_df = results_df[['subject_id', 'n_meals', 'n_test', 'group']].copy()
personal_df['MAE'] = results_df[f'{BEST_MODEL}_MAE']
personal_df['MAPE'] = results_df[f'{BEST_MODEL}_MAPE']
personal_df['Precision_Mean'] = results_df[f'{BEST_MODEL}_Precision']
personal_df = personal_df.sort_values('MAPE')

print(f"\n=== {BEST_MODEL} 개인별 결과 ===")
print(personal_df.to_string(index=False))
print(f"\n전체 평균  MAE={personal_df['MAE'].mean():.1f} mg/dL  MAPE={personal_df['MAPE'].mean():.1f}%  Precision Mean={personal_df['Precision_Mean'].mean():.1f}%")

### 개인별 Precision Mean / MAE 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = personal_df['Precision_Mean'].apply(lambda x: '#278C6C' if x >= 80 else '#C9422B')
ax.bar(range(len(personal_df)), personal_df['Precision_Mean'].values, color=colors)
ax.set_xticks(range(len(personal_df)))
ax.set_xticklabels(personal_df['subject_id'].values, rotation=90, fontsize=7)
ax.axhline(80, color='red', linestyle='--', alpha=0.5, label='80%')
ax.set_ylabel('Precision Mean (%)')
ax.set_title(f'참가자별 예측 정밀도 ({BEST_MODEL})')
ax.legend()

ax = axes[1]
ax.bar(range(len(personal_df)), personal_df['MAE'].values, color='#4A7FB5')
ax.set_xticks(range(len(personal_df)))
ax.set_xticklabels(personal_df['subject_id'].values, rotation=90, fontsize=7)
ax.set_ylabel('MAE (mg/dL)')
ax.set_title(f'참가자별 MAE ({BEST_MODEL})')

plt.tight_layout()
plt.savefig('personal_model_precision.png', dpi=150)
plt.show()

## 4. 콜드스타트 모델 대비 개인화 이득 비교

In [ ]:
coldstart_model = joblib.load(CONFIG["coldstart_model_path"])
coldstart_group_cols = joblib.load(
    os.path.join(CONFIG["cgmacros_dir"], 'final_model_group_columns.pkl')
)
COLDSTART_FEATURES = NUTRIENTS + ['baseline'] + coldstart_group_cols

comparison = []
for _, row in personal_df.iterrows():
    subj = row['subject_id']
    g = meals_valid[meals_valid['subject_id'] == subj]
    X_personal = g[PERSONAL_FEATURES].values
    y = g[TARGET].values
    n = len(y)

    group_dummies = pd.get_dummies(g['group'], prefix='group')
    for col in coldstart_group_cols:
        if col not in group_dummies.columns:
            group_dummies[col] = 0
    X_cold = pd.concat([g[NUTRIENTS + ['baseline']].reset_index(drop=True),
                        group_dummies[coldstart_group_cols].reset_index(drop=True)], axis=1).values

    indices = np.arange(n)
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)
    y_test = y[test_idx]

    cold_preds = coldstart_model.predict(X_cold[test_idx])
    cold_mape = np.mean(np.abs((y_test - cold_preds) / y_test)) * 100
    cold_mae = mean_absolute_error(y_test, cold_preds)

    comparison.append({
        'subject_id': subj,
        'n_meals': row['n_meals'],
        'group': row['group'],
        'coldstart_MAE': round(cold_mae, 1),
        'coldstart_MAPE': round(cold_mape, 1),
        'coldstart_Precision': round(100 - cold_mape, 1),
        'personal_MAE': row['MAE'],
        'personal_MAPE': row['MAPE'],
        'personal_Precision': row['Precision_Mean'],
        'MAPE_gain': round(cold_mape - row['MAPE'], 1),
    })

comp_df = pd.DataFrame(comparison).sort_values('MAPE_gain', ascending=False)
print(comp_df[['subject_id','n_meals','group','coldstart_MAPE','personal_MAPE','MAPE_gain']].to_string(index=False))
print(f"\n개인화가 유리한 참가자 (MAPE_gain > 0): {(comp_df['MAPE_gain'] > 0).sum()}명 / {len(comp_df)}명")
print(f"평균 콜드스타트 Precision: {comp_df['coldstart_Precision'].mean():.1f}%")
print(f"평균 개인화 Precision:     {comp_df['personal_Precision'].mean():.1f}%")

fig, ax = plt.subplots(figsize=(12, 4))
x = range(len(comp_df))
ax.bar(x, comp_df['MAPE_gain'].values,
       color=comp_df['MAPE_gain'].apply(lambda v: '#278C6C' if v > 0 else '#C9422B'))
ax.set_xticks(x)
ax.set_xticklabels(comp_df['subject_id'].values, rotation=90, fontsize=7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('MAPE 이득 (콜드스타트 − 개인화, %p)')
ax.set_title(f'콜드스타트 대비 개인화 모델 MAPE 이득 ({BEST_MODEL})')
plt.tight_layout()
plt.savefig('personal_vs_coldstart.png', dpi=150)
plt.show()

## 5. 전환 기준 분석 — 끼니 수 · 진단군별

In [ ]:
comp_df['meal_bin'] = pd.cut(comp_df['n_meals'], bins=[0, 15, 25, 35, 50, 100],
                             labels=['10-15', '16-25', '26-35', '36-50', '50+'])

bin_summary = comp_df.groupby('meal_bin', observed=True).agg(
    n_subjects=('subject_id', 'count'),
    avg_MAPE_gain=('MAPE_gain', 'mean'),
    pct_gain_positive=('MAPE_gain', lambda x: (x > 0).mean() * 100),
    avg_personal_MAPE=('personal_MAPE', 'mean'),
    avg_coldstart_MAPE=('coldstart_MAPE', 'mean'),
).round(1)
print("끼니 수 구간별:")
print(bin_summary.to_string())

print()
group_eff = comp_df.groupby('group').agg(
    n_subjects=('subject_id', 'count'),
    avg_MAPE_gain=('MAPE_gain', 'mean'),
    pct_gain_positive=('MAPE_gain', lambda x: (x > 0).mean() * 100),
    avg_personal_MAPE=('personal_MAPE', 'mean'),
    avg_coldstart_MAPE=('coldstart_MAPE', 'mean'),
).round(1)
print("진단군별:")
print(group_eff.to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
bin_summary['avg_MAPE_gain'].plot(kind='bar', ax=axes[0],
    color=bin_summary['avg_MAPE_gain'].apply(lambda v: '#278C6C' if v > 0 else '#C9422B'))
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_ylabel('평균 MAPE 이득 (%p)'); axes[0].set_title('끼니 수 구간별 개인화 이득')
axes[0].tick_params(axis='x', rotation=0)

bin_summary['pct_gain_positive'].plot(kind='bar', ax=axes[1], color='#4A7FB5')
axes[1].axhline(50, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('개인화 유리 비율(%)')
axes[1].set_title('끼니 수 구간별 개인화가 유리한 비율')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('personal_transition_threshold.png', dpi=150)
plt.show()

## 6. 개인별 모델 저장

콜드스타트 대비 MAPE가 낮은(개인화가 유리한) 참가자만 전체 데이터로 재학습해서 저장한다.
선정된 단일 모델을 모든 참가자에 동일하게 적용한다.

In [ ]:
eligible_for_save = comp_df[comp_df['MAPE_gain'] > 0]
print(f"저장 대상: {len(eligible_for_save)}명 (모델: {BEST_MODEL})")

model_registry = []
for _, row in eligible_for_save.iterrows():
    subj = row['subject_id']
    g = meals_valid[meals_valid['subject_id'] == subj]
    X = g[PERSONAL_FEATURES].values
    y = g[TARGET].values
    n = len(X)

    if BEST_MODEL == 'LightGBM':
        m = make_lgbm(n)
    elif BEST_MODEL == 'XGBoost':
        m = make_xgb(n)
    else:
        m = make_ridge()

    m.fit(X, y)
    save_path = os.path.join(MODEL_DIR, f'{subj}.pkl')
    joblib.dump(m, save_path)

    model_registry.append({
        'subject_id': subj,
        'model_type': BEST_MODEL,
        'n_meals_trained': n,
        'test_mape': row['personal_MAPE'],
        'precision_mean': round(100 - row['personal_MAPE'], 1),
        'mape_gain_vs_coldstart': row['MAPE_gain'],
        'model_path': save_path,
    })

registry_df = pd.DataFrame(model_registry)
registry_df.to_csv(os.path.join(MODEL_DIR, 'model_registry.csv'), index=False, encoding='utf-8-sig')
print(f"\n저장 완료: {MODEL_DIR}/")
print(registry_df[['subject_id','model_type','n_meals_trained','test_mape','precision_mean','mape_gain_vs_coldstart']].to_string(index=False))

## 7. 추론 함수 — 콜드스타트 / 개인화 자동 전환

In [ ]:
def predict_with_fallback(subject_id, baseline, carbs, protein, fat, fiber,
                          diagnosis_group='건강군',
                          personal_dir=MODEL_DIR):
    """개인 모델이 있으면 사용하고, 없으면 콜드스타트 모델로 폴백한다."""
    personal_path = os.path.join(personal_dir, f'{subject_id}.pkl')

    if os.path.exists(personal_path):
        model = joblib.load(personal_path)
        x = np.array([[carbs, protein, fat, fiber, baseline]])
        pred_peak = model.predict(x)[0]
        source = '개인화'
    else:
        feat = {c: 0 for c in COLDSTART_FEATURES}
        feat['carbs_g'] = carbs
        feat['protein_g'] = protein
        feat['fat_g'] = fat
        feat['fiber_g'] = fiber
        feat['baseline'] = baseline
        group_col = f'group_{diagnosis_group}'
        if group_col in feat:
            feat[group_col] = 1
        x = np.array([[feat[c] for c in COLDSTART_FEATURES]])
        pred_peak = coldstart_model.predict(x)[0]
        source = '콜드스타트'

    return {
        'model_source': source,
        'predicted_peak': round(float(pred_peak), 1),
        'predicted_delta': round(float(pred_peak - baseline), 1),
    }


# 테스트
if len(registry_df) > 0:
    test_subj = registry_df.iloc[0]['subject_id']
    result = predict_with_fallback(test_subj, baseline=120, carbs=50, protein=20, fat=15, fiber=3)
    print(f"[{test_subj}] {result}")

result = predict_with_fallback('NEW_USER', baseline=120, carbs=50, protein=20, fat=15, fiber=3,
                                diagnosis_group='건강군')
print(f"[신규 사용자] {result}")